In [0]:
# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Silver
# Notebook        : Silver_suppliers
# Source          : suppliers.csv
# Target          : procurement.silver.silver_suppliers
#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook loads Cleaned suppliers master data into the Silver layer.
# It validates the source data, separates duplicate records, adds audit columns, and stores the results as Delta tables.

# ==============================================================================
# Business Objective
# ==============================================================================
#
# Read employee data from the Bronze layer, apply data cleansing,standardization, and business rules to create a trusted Silver Delta table.
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
# Import Libraries and Widgets
from pyspark.sql import DataFrame
from pyspark.sql.functions import (col, lit,current_timestamp,when,count,trim,coalesce,initcap,lower,upper,regexp_replace,to_date)
from pyspark.sql.types import (StructType, StructField, StringType,IntegerType,DoubleType,DecimalType)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import try_to_timestamp

In [0]:
# ============================================================
# Read Bronze Sippliers Table
# ============================================================

bronze_suppliers_df = read_delta(BRONZE_SUPPLIERS)

preview(bronze_suppliers_df,"Bronze suppliers")

In [0]:
#============================================
# Create Sliver DataFrame
#===========================================
silver_suppliers_df = bronze_suppliers_df

In [0]:
# ============================================================
# Apply business transformations
# ============================================================

# Standardize suppliers_status as Matched
silver_suppliers_df = (silver_suppliers_df
    .withColumn("supplier_id", trim(col("supplier_id")))
    .withColumn("supplier_name", initcap(trim(col("supplier_name"))))
    .withColumn("category", upper(trim(col("category"))))
    .withColumn("country", upper(trim(col("country"))))
    .withColumn("city", initcap(trim(col("city"))))
    .withColumn("contact_email", lower(trim(col("contact_email"))))
    .withColumn("phone", regexp_replace(trim(col("phone")),"[ -]",""))
    .withColumn("supplier_rating", round(col("supplier_rating"),1))
    .withColumn("is_active", when(upper(trim(col("is_active"))) == "YES" ,True)
                              .when(upper(trim(col("is_active"))) == "NO" ,False)
                              .otherwise(col("is_active")))
    .withColumn("onboarded_date", to_date(col("onboarded_date"),"dd-MM-yyyy"))
    .withColumn("payment_terms_default", when(upper(trim(col("payment_terms_default"))) == "NET30" ,"NET 30").otherwise(trim(col("payment_terms_default"))))
)



In [0]:
# ============================================================
# Identify invalid supplier records
# NULL & Blank invoice_id
# NULL: supplier_name,category country,city,contact_email,phone,supplier,rating,is_active,onboarding_date,payment_terms_default
# ============================================================
invalid_suppliers = silver_suppliers_df.filter(
    (col("supplier_id").isNull() | (trim(col("supplier_id")) == ""))
    | (col("supplier_name").isNull() | (trim(col("supplier_name")) == ""))
    | (col("category").isNull() | (trim(col("category")) == ""))
    | (col("country").isNull() | (trim(col("country")) == ""))
    | (col("city").isNull() | (trim(col("city")) == ""))
    | (col("contact_email").isNull() | (trim(col("contact_email")) == ""))
    | (col("phone").isNull() | (trim(col("phone")) == ""))
    | col("supplier_rating").isNull()
    | col("is_active").isNull()
    | col("onboarded_date").isNull() 
    | (col("payment_terms_default").isNull() | (trim(col("payment_terms_default")) == ""))
)

print(f"Invalid Supplier Records:{invalid_suppliers.count()}")
display(invalid_suppliers)

In [0]:
# ============================================================
# Add Audit Metadata
# ============================================================

invalid_suppliers = (invalid_suppliers.withColumn("audit_timestamp",current_timestamp())
    .withColumn("source_table",lit("Suppliers"))
    .withColumn("pipeline_layer",lit("Silver"))
    .withColumn("issue_type",lit("Invalid Record")))

display(invalid_suppliers)

In [0]:
# ============================================================
# Write Invalid Suppliers to Audit Table
# ============================================================

if invalid_suppliers.count() > 0:
    write_delta(invalid_suppliers,AUDIT_INVALID_SUPPLIERS,mode="overwrite")
    print("Invalid supplier records written.")
else:
    print("No invalid supplier records found.")

In [0]:
# ============================================================
# Remove Invalid Records
# ============================================================

silver_suppliers_df = silver_suppliers_df.filter(

    col("supplier_id").isNotNull() & (trim(col("supplier_id")) != "") &

    col("supplier_name").isNotNull() & (trim(col("supplier_name")) != "") &

    col("category").isNotNull() & (trim(col("category")) != "") &

    col("country").isNotNull() & (trim(col("country")) != "") &

    col("city").isNotNull() & (trim(col("city")) != "") &

    col("contact_email").isNotNull() & (trim(col("contact_email")) != "") &

    col("phone").isNotNull() & (trim(col("phone")) != "") &

    col("supplier_rating").isNotNull() & col("is_active").isNotNull() &

    col("onboarded_date").isNotNull() & col("payment_terms_default").isNotNull() & (trim(col("payment_terms_default")) != "")
)

display(silver_suppliers_df)

In [0]:
# ============================================================
# Remove Duplicate Suppliers
# ============================================================

window_spec = Window.partitionBy("supplier_id").orderBy(col("onboarded_date").desc())

silver_suppliers_df = (silver_suppliers_df.withColumn("row_num",row_number().over(window_spec))
                      .filter(col("row_num") == 1).drop("row_num"))
preview(silver_suppliers_df,"Silver Suppliers")


In [0]:
# ============================================================
# Write Silver Delta Table
# ============================================================

silver_suppliers_df = (silver_suppliers_df
                         .withColumn("silver_load_timestamp", current_timestamp())
                         .withColumn("pipeline_layer", lit("Silver"))
)

In [0]:
# ============================================================
# Write Silver Delta Table
# ============================================================

write_delta(df=silver_suppliers_df,table_name=SILVER_SUPPLIERS)

In [0]:
# ============================================================
# Validate Summary
# ============================================================

bronze_count = bronze_suppliers_df.count()
invalid_count = invalid_suppliers.count()
silver_count = silver_suppliers_df.count()

duplicate_removed = bronze_count - invalid_count - silver_count

print("=" * 60)
print("Silver Suppliers Load Completed Successfully")
print("=" * 60)

print(f"Bronze Records            : {bronze_count}")
print(f"Invalid Records Removed   : {invalid_count}")
print(f"Duplicate Records Removed : {duplicate_removed}")
print(f"Silver Records            : {silver_count}")